# JobCompass — Notebook 03: Embedding Generation

## What this does
Turns every candidate and job into a dense vector using **Qwen3-Embedding-0.6B**.

These vectors are used in NB05 for semantic (ANN) search — finding candidates/jobs that are
semantically similar even when they don't share exact keywords.

## Model choice: Qwen3-Embedding-0.6B
- #1 on MTEB retrieval leaderboard (May 2025)
- 0.6B params → fits in ~1.5GB VRAM on your RTX 4060 8GB
- 2048-dimensional output (richer than bge-large's 768)
- Instruction-aware: you tell it what the text represents

## Resume safety
Output files are opened in **append mode**. A checkpoint JSON tracks which IDs
have been written. On resume, those IDs are skipped. No data goes missing.

## Instructions to run
1. Ensure NB02 is complete (`candidates.jsonl` and `jobs.jsonl` exist)
2. No server needed — model loads directly in this notebook
3. Run all cells top to bottom
4. If interrupted: just rerun — it picks up from where it stopped

## Output
- `data/processed/candidates_embedded.jsonl` — candidates with `description_vector` field
- `data/processed/jobs_embedded.jsonl` — jobs with `description_vector` field
- `data/processed/nb03_progress.json` — checkpoint (safe to delete to start fresh)

In [1]:
# Cell 0 — Install dependencies
import subprocess, sys

packages = [
    "transformers>=4.51.0",   # Qwen3 embedding support added in 4.51
    "torch",
    "orjson",
    "tqdm",
    "numpy",
]

for pkg in packages:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    print(f"  {'OK' if r.returncode == 0 else 'FAIL'} {pkg}")
print("Done.")

  OK transformers>=4.51.0
  OK torch
  OK orjson
  OK tqdm
  OK numpy
Done.


In [2]:
# Cell 1 — Configuration
import os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
PROCESSED_DIR    = "../data/processed"
CANDIDATES_IN    = f"{PROCESSED_DIR}/candidates.jsonl"
JOBS_IN          = f"{PROCESSED_DIR}/jobs.jsonl"
CANDIDATES_OUT   = f"{PROCESSED_DIR}/candidates_embedded.jsonl"
JOBS_OUT         = f"{PROCESSED_DIR}/jobs_embedded.jsonl"
PROGRESS_FILE    = f"{PROCESSED_DIR}/nb03_progress.json"

# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_NAME       = "Qwen/Qwen3-Embedding-0.6B"
EMBEDDING_DIM    = 1024     # ← fixed: 0.6B outputs 1024 dims (4B outputs 2048)
MAX_SEQ_LENGTH   = 512

CANDIDATE_INSTRUCTION = "Represent this resume for job matching"
JOB_INSTRUCTION       = "Represent this job description for candidate matching"

# ── Batching ───────────────────────────────────────────────────────────────────
ENCODE_BATCH_SIZE = 128
CHECKPOINT_EVERY  = 128

# ── Cache ──────────────────────────────────────────────────────────────────────
os.environ["HF_HOME"] = "D:/huggingface_cache"

# Verify inputs
for p in [CANDIDATES_IN, JOBS_IN]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Not found: {p} — run NB02 first.")

os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Config ready.")
print(f"  Model         : {MODEL_NAME}")
print(f"  Embedding dim : {EMBEDDING_DIM}")
print(f"  Batch size    : {ENCODE_BATCH_SIZE}")
print(f"  Checkpoint    : every {CHECKPOINT_EVERY} records")

Config ready.
  Model         : Qwen/Qwen3-Embedding-0.6B
  Embedding dim : 1024
  Batch size    : 128
  Checkpoint    : every 128 records


In [3]:
# Cell 2 — GPU check + auto-tune batch size
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

if device == "cuda":
    torch.cuda.empty_cache()  # clear any leftover allocations before measuring
    props    = torch.cuda.get_device_properties(0)
    vram_gb  = props.total_memory / 1024**3
    free_gb  = (props.total_memory - torch.cuda.memory_reserved(0)) / 1024**3
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {vram_gb:.1f} GB total  /  {free_gb:.1f} GB free")

    # Tune batch size based on FREE (not total) VRAM.
    # Qwen3-0.6B weights ~1.2GB in fp16.
    # Each 512-token sequence in a batch costs ~20MB activation → 32 seqs ≈ 640MB.
    # Keep total under ~70% of free VRAM to avoid OOM.
    if free_gb < 2.5:
        ENCODE_BATCH_SIZE = 8
        print(f"⚠ Very low free VRAM → ENCODE_BATCH_SIZE = {ENCODE_BATCH_SIZE}")
    elif free_gb < 4.0:
        ENCODE_BATCH_SIZE = 16
        print(f"⚠ Low free VRAM → ENCODE_BATCH_SIZE = {ENCODE_BATCH_SIZE}")
    elif free_gb < 5.5:
        ENCODE_BATCH_SIZE = 32
        print(f"ℹ Moderate VRAM → ENCODE_BATCH_SIZE = {ENCODE_BATCH_SIZE}")
    elif free_gb < 7.0:
        ENCODE_BATCH_SIZE = 64
        print(f"✓ Good VRAM → ENCODE_BATCH_SIZE = {ENCODE_BATCH_SIZE}")
    else:
        ENCODE_BATCH_SIZE = 128
        print(f"✓ Ample VRAM → ENCODE_BATCH_SIZE = {ENCODE_BATCH_SIZE}")
else:
    ENCODE_BATCH_SIZE = 32
    print("⚠ No GPU — CPU mode (slower but works)")

print(f"\nFinal ENCODE_BATCH_SIZE : {ENCODE_BATCH_SIZE}")

Device : cuda
GPU    : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM   : 8.0 GB total  /  8.0 GB free
✓ Ample VRAM → ENCODE_BATCH_SIZE = 128

Final ENCODE_BATCH_SIZE : 128


In [4]:
# Cell 3 — Load Qwen3-Embedding-0.6B
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModel

print(f"Loading {MODEL_NAME} ...")
print("(Downloads ~1.2GB on first run, cached after that)")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=os.environ["HF_HOME"],
)

embed_model = AutoModel.from_pretrained(
    MODEL_NAME,
    cache_dir   = os.environ["HF_HOME"],
    torch_dtype = torch.float16,
    device_map  = "auto",
)
embed_model.eval()

# Disable KV-cache and gradient computation globally
torch.backends.cuda.enable_flash_sdp(False)   # avoids some OOM patterns on older drivers


def last_token_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_state[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size       = last_hidden_state.shape[0]
        return last_hidden_state[
            torch.arange(batch_size, device=last_hidden_state.device),
            sequence_lengths,
        ]


def encode_texts(texts: list[str], instruction: str) -> np.ndarray:
    """
    Encode texts in micro-batches to avoid OOM.
    Falls back to smaller sub-batches automatically if an OOM is hit.
    Returns L2-normalised float32 vectors, shape (N, EMBEDDING_DIM).
    """
    formatted = [
        f"Instruct: {instruction}\nQuery: {t.strip()}" if t.strip() else instruction
        for t in texts
    ]

    all_embeddings = []
    micro_batch_size = ENCODE_BATCH_SIZE  # start with auto-tuned size

    i = 0
    while i < len(formatted):
        chunk = formatted[i : i + micro_batch_size]
        try:
            inputs = tokenizer(
                chunk,
                max_length     = MAX_SEQ_LENGTH,
                padding        = True,
                truncation     = True,
                return_tensors = "pt",
            ).to(embed_model.device)

            with torch.inference_mode():   # lighter than no_grad: frees more memory
                outputs = embed_model(**inputs)

            embeddings = last_token_pool(outputs.last_hidden_state, inputs["attention_mask"])
            embeddings = F.normalize(embeddings, p=2, dim=1)
            all_embeddings.append(embeddings.cpu().float().numpy())

            # Explicitly delete tensors to release VRAM immediately
            del inputs, outputs, embeddings
            if device == "cuda":
                torch.cuda.empty_cache()

            i += micro_batch_size  # advance only on success

        except RuntimeError as e:
            if "out of memory" in str(e).lower() and micro_batch_size > 1:
                # Halve the batch size and retry the same chunk
                if device == "cuda":
                    torch.cuda.empty_cache()
                micro_batch_size = max(1, micro_batch_size // 2)
                print(f"\n⚠ OOM — reducing micro_batch_size to {micro_batch_size} and retrying...")
            else:
                raise  # non-OOM error or batch=1 already → give up

    return np.vstack(all_embeddings)


# Smoke test
if device == "cuda":
    torch.cuda.empty_cache()

test = encode_texts(["software engineer python django"], CANDIDATE_INSTRUCTION)
assert test.shape == (1, EMBEDDING_DIM), f"Expected (1, {EMBEDDING_DIM}), got {test.shape}"
norm = np.linalg.norm(test[0])
assert 0.99 < norm < 1.01, f"Expected unit norm, got {norm:.4f}"
print(f"Model loaded on {embed_model.device} ✓")
print(f"Smoke test: shape={test.shape}, norm={norm:.4f} ✓")

Loading Qwen/Qwen3-Embedding-0.6B ...
(Downloads ~1.2GB on first run, cached after that)


`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded on cuda:0 ✓
Smoke test: shape=(1, 1024), norm=1.0002 ✓


In [5]:
# Cell 4 — Load checkpoint + input data
import orjson
from pathlib import Path


def load_checkpoint() -> dict:
    if Path(PROGRESS_FILE).exists():
        with open(PROGRESS_FILE, "rb") as f:
            ckpt = orjson.loads(f.read())
        print(f"Checkpoint loaded.")
        print(f"  Candidates done : {len(ckpt.get('candidates_done', []))}")
        print(f"  Jobs done       : {len(ckpt.get('jobs_done', []))}")
        return ckpt
    else:
        print("No checkpoint found — starting fresh.")
        return {"candidates_done": [], "jobs_done": []}


def save_checkpoint(ckpt: dict):
    with open(PROGRESS_FILE, "wb") as f:
        f.write(orjson.dumps(ckpt))


checkpoint = load_checkpoint()

# Load candidates
candidates = []
with open(CANDIDATES_IN, "rb") as f:
    for line in f:
        if line.strip():
            candidates.append(orjson.loads(line))
print(f"\nCandidates loaded : {len(candidates):,}")

# Load jobs
jobs = []
with open(JOBS_IN, "rb") as f:
    for line in f:
        if line.strip():
            jobs.append(orjson.loads(line))
print(f"Jobs loaded       : {len(jobs):,}")

# Quick check on embedding_text coverage
cand_empty = sum(1 for c in candidates if not (c.get("embedding_text") or "").strip())
job_empty  = sum(1 for j in jobs if not (j.get("embedding_text") or "").strip())
print(f"\nMissing embedding_text:")
print(f"  Candidates : {cand_empty} (fallback to raw_text[:800])")
print(f"  Jobs       : {job_empty}  (fallback to raw_text[:800])")

No checkpoint found — starting fresh.

Candidates loaded : 2,887
Jobs loaded       : 5,000

Missing embedding_text:
  Candidates : 267 (fallback to raw_text[:800])
  Jobs       : 0  (fallback to raw_text[:800])


In [6]:
# Cell 5 — Core encode-and-write function
#
# How resume safety works:
#   - Output file opened in append mode ("ab") — existing data never touched
#   - done_ids loaded from checkpoint at start — these records are skipped
#   - After each batch is written to disk: checkpoint is updated immediately
#   - If process dies mid-batch: that batch is lost (max CHECKPOINT_EVERY records)
#     but everything before it is safe
#   - On next run: done_ids excludes the lost batch → it gets re-encoded

import time
from tqdm.auto import tqdm


def encode_and_write(
    records     : list[dict],
    id_field    : str,
    text_field  : str,
    instruction : str,
    output_path : str,
    done_ids    : set,
    ckpt_key    : str,
    checkpoint  : dict,
) -> int:
    """
    Encodes all records not in done_ids and appends them to output_path.

    Parameters:
        records     : list of dicts from candidates.jsonl or jobs.jsonl
        id_field    : field name of the unique ID ('candidate_id' or 'job_id')
        text_field  : field to encode ('embedding_text')
        instruction : task instruction for Qwen3 (different for candidates vs jobs)
        output_path : where to append output records (with description_vector added)
        done_ids    : IDs already written in a previous run — skip these
        ckpt_key    : key in checkpoint dict to update ('candidates_done'/'jobs_done')
        checkpoint  : the checkpoint dict (mutated in-place and saved after each batch)

    Returns: total number of records written this run.
    """
    # Filter to only records not yet done
    remaining = [r for r in records if r[id_field] not in done_ids]
    skipped   = len(records) - len(remaining)

    print(f"  Total          : {len(records):,}")
    print(f"  Already done   : {skipped:,}  (skipped)")
    print(f"  To encode      : {len(remaining):,}")

    if not remaining:
        print("  Nothing to do — all records already embedded.")
        return 0

    written  = 0
    t_start  = time.time()

    out_fh = open(output_path, "ab")   # append mode — never overwrites existing data

    try:
        pbar = tqdm(total=len(remaining), desc="  Encoding", unit="rec")

        for batch_start in range(0, len(remaining), ENCODE_BATCH_SIZE):
            batch = remaining[batch_start : batch_start + ENCODE_BATCH_SIZE]

            # Get text for each record — fallback to raw_text if embedding_text empty
            texts = []
            for rec in batch:
                text = (rec.get(text_field) or "").strip()
                if not text:
                    text = (rec.get("raw_text") or "")[:800].strip()
                texts.append(text)

            # Encode the batch
            vectors = encode_texts(texts, instruction)   # shape (batch_size, EMBEDDING_DIM)

            # Write each record with its vector appended
            batch_ids = []
            for rec, vec in zip(batch, vectors):
                out_rec = {**rec, "description_vector": vec.tolist()}
                out_fh.write(orjson.dumps(out_rec) + b"\n")
                batch_ids.append(rec[id_field])
                written += 1

            # Flush to disk and update checkpoint
            # This happens every batch — if we crash after this point,
            # these records are safe and will be skipped on next run
            out_fh.flush()
            checkpoint[ckpt_key].extend(batch_ids)
            done_ids.update(batch_ids)
            save_checkpoint(checkpoint)

            pbar.update(len(batch))

            # Periodically free CUDA memory to prevent fragmentation
            if device == "cuda" and batch_start % (ENCODE_BATCH_SIZE * 8) == 0:
                torch.cuda.empty_cache()

        pbar.close()

    finally:
        # Always close the file even if an exception occurs mid-run
        out_fh.close()

    elapsed = time.time() - t_start
    print(f"  Written  : {written:,}")
    print(f"  Time     : {elapsed:.1f}s  ({written / max(elapsed, 0.001):.0f} rec/s)")
    print(f"  Output   : {output_path}")
    return written


print("encode_and_write() defined.")

encode_and_write() defined.


In [7]:
# Cell 6 — Encode candidates
print("=" * 55)
print("CANDIDATES")
print("=" * 55)

done_cand_ids = set(checkpoint.get("candidates_done", []))

n_cands = encode_and_write(
    records     = candidates,
    id_field    = "candidate_id",
    text_field  = "embedding_text",
    instruction = CANDIDATE_INSTRUCTION,
    output_path = CANDIDATES_OUT,
    done_ids    = done_cand_ids,
    ckpt_key    = "candidates_done",
    checkpoint  = checkpoint,
)

print(f"\nCandidates done. Total in output: {len(checkpoint['candidates_done']):,}")

CANDIDATES
  Total          : 2,887
  Already done   : 0  (skipped)
  To encode      : 2,887


  Encoding: 100%|██████████| 2887/2887 [02:04<00:00, 23.10rec/s]

  Written  : 2,887
  Time     : 125.0s  (23 rec/s)
  Output   : ../data/processed/candidates_embedded.jsonl

Candidates done. Total in output: 2,887


In [8]:
# Cell 7 — Encode jobs
print("=" * 55)
print("JOBS")
print("=" * 55)

done_job_ids = set(checkpoint.get("jobs_done", []))

n_jobs = encode_and_write(
    records     = jobs,
    id_field    = "job_id",
    text_field  = "embedding_text",
    instruction = JOB_INSTRUCTION,
    output_path = JOBS_OUT,
    done_ids    = done_job_ids,
    ckpt_key    = "jobs_done",
    checkpoint  = checkpoint,
)

print(f"\nJobs done. Total in output: {len(checkpoint['jobs_done']):,}")

JOBS
  Total          : 5,000
  Already done   : 0  (skipped)
  To encode      : 5,000


  Encoding: 100%|██████████| 5000/5000 [01:06<00:00, 75.67rec/s]

  Written  : 5,000
  Time     : 66.1s  (76 rec/s)
  Output   : ../data/processed/jobs_embedded.jsonl

Jobs done. Total in output: 5,000


In [9]:
# Cell 8 — Validate output
# Checks every record has a correctly shaped, finite, unit-norm vector.

import numpy as np

def validate(path: str, id_field: str):
    records = []
    with open(path, "rb") as f:
        for line in f:
            if line.strip():
                records.append(orjson.loads(line))

    issues = {"missing": 0, "wrong_dim": 0, "nan_inf": 0, "non_unit": 0}

    for rec in records:
        vec = rec.get("description_vector")
        if vec is None:
            issues["missing"] += 1
            continue
        arr = np.array(vec, dtype=np.float32)
        if len(arr) != EMBEDDING_DIM:
            issues["wrong_dim"] += 1
            continue
        if np.any(~np.isfinite(arr)):
            issues["nan_inf"] += 1
            continue
        if not (0.98 < np.linalg.norm(arr) < 1.02):
            issues["non_unit"] += 1

    print(f"{path}")
    print(f"  Total records  : {len(records):,}")
    for k, v in issues.items():
        status = "✓" if v == 0 else "✗"
        print(f"  {k:12s}   : {v}  {status}")
    print()
    return records


print("Validation:")
print()
validated_cands = validate(CANDIDATES_OUT, "candidate_id")
validated_jobs  = validate(JOBS_OUT,       "job_id")

Validation:

../data/processed/candidates_embedded.jsonl
  Total records  : 2,887
  missing        : 0  ✓
  wrong_dim      : 0  ✓
  nan_inf        : 0  ✓
  non_unit       : 0  ✓

../data/processed/jobs_embedded.jsonl
  Total records  : 5,000
  missing        : 0  ✓
  wrong_dim      : 0  ✓
  nan_inf        : 0  ✓
  non_unit       : 0  ✓



In [10]:
# Cell 9 — Embedding space sanity check
# Shows top candidate-job matches. If a Python developer matches Python Developer jobs,
# your embedding space is working. If matches look random, something is wrong.

import numpy as np
import random

SAMPLE_N = min(100, len(validated_cands), len(validated_jobs))

sample_cands = random.sample(validated_cands, SAMPLE_N)
sample_jobs  = random.sample(validated_jobs,  SAMPLE_N)

cand_vecs = np.array([c["description_vector"] for c in sample_cands], dtype=np.float32)
job_vecs  = np.array([j["description_vector"] for j in sample_jobs],  dtype=np.float32)

# Cosine similarity matrix (unit norm → dot product = cosine)
sim_matrix = cand_vecs @ job_vecs.T

print(f"Cross-similarity stats (sample n={SAMPLE_N}):")
print(f"  Mean  : {sim_matrix.mean():.4f}")
print(f"  Std   : {sim_matrix.std():.4f}")
print(f"  Range : [{sim_matrix.min():.3f}, {sim_matrix.max():.3f}]")
print()
print(f"Top matches (candidate title → best matching job):")
print(f"{'Candidate':35s}  {'Best matching job':35s}  {'Sim':>6}")
print("-" * 82)

shown = 0
for i in range(len(sample_cands)):
    top_j   = int(np.argmax(sim_matrix[i]))
    c_title = str(sample_cands[i].get("current_title") or "unknown")[:34]
    j_title = str(sample_jobs[top_j].get("normalized_title") or "unknown")[:34]
    sim     = sim_matrix[i, top_j]
    # Only print if similarity is reasonable (> 0.4) to show meaningful matches
    if sim > 0.4:
        print(f"{c_title:35s}  {j_title:35s}  {sim:>6.4f}")
        shown += 1
    if shown >= 10:
        break

if shown == 0:
    print("No matches above 0.4 threshold in sample — showing top 5 regardless:")
    top_pairs = sorted(
        [(i, int(np.argmax(sim_matrix[i])), float(np.max(sim_matrix[i])))
         for i in range(len(sample_cands))],
        key=lambda x: -x[2]
    )[:5]
    for i, j, s in top_pairs:
        c_title = str(sample_cands[i].get("current_title") or "unknown")[:34]
        j_title = str(sample_jobs[j].get("normalized_title") or "unknown")[:34]
        print(f"{c_title:35s}  {j_title:35s}  {s:>6.4f}")

Cross-similarity stats (sample n=100):
  Mean  : 0.3708
  Std   : 0.0629
  Range : [0.171, 0.707]

Top matches (candidate title → best matching job):
Candidate                            Best matching job                       Sim
----------------------------------------------------------------------------------
unknown                              Logistics Analyst                    0.5311
unknown                              Database Analyst                     0.4704
unknown                              Electrical Designer                  0.5126
Public Relations officer             Digital Marketing Specialist         0.4886
unknown                              Outside Sales Representative         0.4430
Digital Media - Kihn LLC             Marketing Director                   0.5695
unknown                              Database Analyst                     0.4897
Dot Net Developer III                Back-End Developer                   0.4916
SENIOR PRINCIPAL SYSTEMS SECURITY    P

In [ ]:
# Cell 10 — Cleanup
import gc

try:
    del embed_model
    del tokenizer
    if device == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    print("Model unloaded — VRAM freed.")
except Exception:
    gc.collect()

print()
print("NB03 complete.")
print(f"  Candidates : {len(validated_cands):>6}  ->  {CANDIDATES_OUT}")
print(f"  Jobs       : {len(validated_jobs):>6}  ->  {JOBS_OUT}")
print()
print("Next: Notebook 04 — OpenSearch Indexing")
print("  Prerequisites:")
print("    - Docker Desktop running")
print("    - Run: docker run -d -p 9200:9200 -e discovery.type=single-node \\")
print("             -e DISABLE_SECURITY_PLUGIN=true \\")
print("             opensearchproject/opensearch:2.13.0")
print("  Input  : candidates_embedded.jsonl + jobs_embedded.jsonl")
print("  Output : two OpenSearch indices with BM25 text + kNN HNSW vector fields")

Model unloaded — VRAM freed.

NB03 complete.
  Candidates :   2887  ->  ../data/processed/candidates_embedded.jsonl
  Jobs       :   5000  ->  ../data/processed/jobs_embedded.jsonl

Next: Notebook 04 — OpenSearch Indexing
  Prerequisites:
    - Docker Desktop running
    - Run: docker run -d -p 9200:9200 -e discovery.type=single-node \
             -e DISABLE_SECURITY_PLUGIN=true \
             opensearchproject/opensearch:2.13.0
  Input  : candidates_embedded.jsonl + jobs_embedded.jsonl
  Output : two OpenSearch indices with BM25 text + kNN HNSW vector fields


: 